# LeetCode 2068 — Check Whether Two Strings are Almost Equivalent bằng Python

Link đề: https://leetcode.com/problems/check-whether-two-strings-are-almost-equivalent/ · Độ khó: **Easy** · Chủ đề: Hash Table, String, Counting

```python
class Solution:
    def checkAlmostEquivalent(self, word1: str, word2: str) -> bool:

```

**Mục tiêu học**

1. Hiểu **tần suất (frequency)** của ký tự và cách so sánh tần suất giữa hai chuỗi.
2. Thành thạo ba công cụ đếm: `str.count`, `collections.Counter`, và **mảng 26 ô** với `ord(ch) - ord('a')`.
3. Nắm mẹo "**một mảng hiệu**": `+1` cho `word1`, `−1` cho `word2`, rồi xét `abs`.
4. Không quên ký tự **chỉ có ở một chuỗi** (gợi ý của đề) — lỗi kinh điển khi chỉ duyệt key của một Counter.
5. Nộp `class Solution`, test đối chiếu các cách.

## 1. Đề bài yêu cầu gì?

Hai chuỗi `word1`, `word2` (cùng độ dài `n`) là **almost equivalent** nếu với **mọi** chữ cái từ `'a'` đến `'z'`, **chênh lệch tần suất** giữa hai chuỗi **không quá 3**.

| Ví dụ | word1 | word2 | Output | Lý do |
|---|---|---|---|---|
| 1 | `"aaaa"` | `"bccb"` | `False` | `a`: 4 vs 0 → chênh 4 > 3 |
| 2 | `"abcdeef"` | `"abaaacc"` | `True` | chênh lớn nhất là 3 (chữ `a`: 1 vs 4) |
| 3 | `"cccddabba"` | `"babababab"` | `True` | chênh lớn nhất là 3 (`b`: 2 vs 5, `c`: 3 vs 0) |

Constraints: `n == len(word1) == len(word2)`, `1 <= n <= 100`, chỉ chữ thường.

Viết dưới dạng công thức: với mọi `c`, `|freq1[c] − freq2[c]| <= 3`. Chữ không có trong chuỗi có tần suất **0** — vẫn phải xét!

In [1]:
w1, w2 = "aaaa", "bccb"
for c in "abc":
    f1, f2 = w1.count(c), w2.count(c)
    print(f"{c}: {f1} vs {f2}, chênh {abs(f1 - f2)}")

a: 4 vs 0, chênh 4
b: 0 vs 2, chênh 2
c: 0 vs 2, chênh 2


## 2. Nền tảng Python

### 2.1. `str.count(c)` — đếm số lần ký tự xuất hiện

Duyệt cả chuỗi → `O(n)` mỗi lần gọi.

In [2]:
print("abaaacc".count("a"), "abaaacc".count("z"))   # 4 0

4 0


### 2.2. `string.ascii_lowercase` — 26 chữ cái

Module chuẩn `string` có sẵn chuỗi `"abcdefghijklmnopqrstuvwxyz"`, tiện để duyệt **mọi** chữ cái.

In [3]:
import string
print(string.ascii_lowercase, len(string.ascii_lowercase))

abcdefghijklmnopqrstuvwxyz 26


### 2.3. `ord` / `chr` và mảng 26 ô

- `ord('a') = 97`, `ord('b') = 98`, ..., liên tiếp nhau.
- `ord(ch) - ord('a')` đổi `'a'..'z'` thành chỉ số `0..25`.
- `chr(97 + k)` làm ngược lại.

Mảng `cnt = [0] * 26`: `cnt[k]` là tần suất của chữ thứ `k`.

In [4]:
for ch in "abz":
    print(ch, ord(ch), ord(ch) - ord('a'))
print(chr(ord('a') + 2))

cnt = [0] * 26
for ch in "abaaacc":
    cnt[ord(ch) - ord('a')] += 1
print(cnt[:5])          # a, b, c, d, e

a 97 0
b 98 1
z 122 25
c
[4, 1, 2, 0, 0]


### 2.4. `Counter` và phép trừ của Counter

`Counter(s)` đếm sẵn. Truy cập key không có trả `0`. **Cẩn thận:** `c1 - c2` của Counter **bỏ các giá trị ≤ 0**, nên nó chỉ cho phần "word1 nhiều hơn".

In [5]:
from collections import Counter
c1, c2 = Counter("abcdeef"), Counter("abaaacc")
print(c1)
print(c2)
print("c1 - c2 =", c1 - c2)      # chỉ giữ phần dương
print("c2 - c1 =", c2 - c1)
print(c1["z"])                   # 0

Counter({'e': 2, 'a': 1, 'b': 1, 'c': 1, 'd': 1, 'f': 1})
Counter({'a': 4, 'c': 2, 'b': 1})
c1 - c2 = Counter({'e': 2, 'd': 1, 'f': 1})
c2 - c1 = Counter({'a': 3, 'c': 1})
0


### 2.5. `all(...)` và `max(...)` với generator

- `all(abs(x) <= 3 for x in diffs)`: mọi chênh lệch ≤ 3?
- `max(abs(x) for x in diffs) <= 3`: tương đương, nhưng `all` dừng sớm.

In [6]:
diffs = [0, -3, 2, 1]
print(all(abs(x) <= 3 for x in diffs), max(abs(x) for x in diffs))

True 3


## 3. Cách 1 — Trực giác: với mỗi chữ cái, đếm ở cả hai chuỗi

Duyệt 26 chữ cái, dùng `count` cho mỗi chuỗi. Đơn giản và **không bao giờ quên** chữ chỉ có ở một chuỗi (vì duyệt đủ 26 chữ).

In [7]:
def almost_equiv_count(word1, word2):
    for c in string.ascii_lowercase:
        if abs(word1.count(c) - word2.count(c)) > 3:
            return False
    return True

print(almost_equiv_count("aaaa", "bccb"))              # False
print(almost_equiv_count("abcdeef", "abaaacc"))        # True
print(almost_equiv_count("cccddabba", "babababab"))    # True

False
True
True


### Chi phí và nút thắt

26 chữ × 2 lần `count` × `n` ký tự = `52n` bước → vẫn `O(n)` (vì 26 là hằng số), nhưng duyệt mỗi chuỗi **26 lần**. Ý tưởng tốt hơn: **duyệt mỗi chuỗi đúng một lần**, đếm tất cả chữ cái cùng lúc vào một cấu trúc (Counter hoặc mảng 26 ô).

**Insight thêm:** ta không cần hai bảng đếm riêng. Chỉ cần **hiệu** `freq1[c] − freq2[c]`: cộng 1 khi gặp `c` trong `word1`, trừ 1 khi gặp trong `word2`. Cuối cùng kiểm tra `abs(diff[c]) <= 3`.

## 4. Cách 2 — Hai Counter, duyệt hợp các key

Phải duyệt **hợp** key của cả hai Counter (`c1.keys() | c2.keys()`), nếu không sẽ bỏ sót chữ chỉ có ở `word2`.

In [8]:
def almost_equiv_counter(word1, word2):
    c1, c2 = Counter(word1), Counter(word2)
    for ch in c1.keys() | c2.keys():          # hợp hai tập key
        if abs(c1[ch] - c2[ch]) > 3:
            return False
    return True

print(almost_equiv_counter("aaaa", "bccb"))
print(almost_equiv_counter("abcdeef", "abaaacc"))
print(almost_equiv_counter("cccddabba", "babababab"))

False
True
True


`c1.keys() | c2.keys()`: `keys()` của dict hỗ trợ phép toán tập hợp như `set`.

## 5. Cách 3 (lời giải chính) — Một mảng hiệu 26 ô

Từng mảnh:

1. `diff = [0] * 26`.
2. Duyệt `word1`: `diff[idx] += 1`. Duyệt `word2`: `diff[idx] -= 1`. (Vì hai chuỗi cùng độ dài, có thể dùng `zip` duyệt song song.)
3. `return all(abs(d) <= 3 for d in diff)`.

In [9]:
def almost_equiv_diff(word1, word2):
    diff = [0] * 26
    for a, b_ in zip(word1, word2):            # cùng độ dài n
        diff[ord(a) - ord('a')] += 1
        diff[ord(b_) - ord('a')] -= 1
    return all(abs(d) <= 3 for d in diff)

print(almost_equiv_diff("aaaa", "bccb"))
print(almost_equiv_diff("abcdeef", "abaaacc"))
print(almost_equiv_diff("cccddabba", "babababab"))

False
True
True


## 6. TRACE từng bước

Ví dụ 2: `word1 = "abcdeef"`, `word2 = "abaaacc"`. Chỉ in các chữ `a..f`.

| bước | ký tự word1 (+1) | ký tự word2 (−1) | diff[a..f] sau bước |
|---|---|---|---|
| 0 | a | a | [0,0,0,0,0,0] |
| 1 | b | b | [0,0,0,0,0,0] |
| 2 | c | a | [−1,0,1,0,0,0] |
| 3 | d | a | [−2,0,1,1,0,0] |
| 4 | e | a | [−3,0,1,1,1,0] |
| 5 | e | c | [−3,0,0,1,2,0] |
| 6 | f | c | [−3,0,−1,1,2,1] |

Lớn nhất `|−3| = 3 ≤ 3` → `True`. Đối chiếu với đề: `a` chênh 3, `c` chênh 1, `d` 1, `e` 2, `f` 1 ✓.

In [10]:
def trace_diff(word1, word2, show="abcdef"):
    diff = [0] * 26
    for step, (a, b_) in enumerate(zip(word1, word2)):
        diff[ord(a) - ord('a')] += 1
        diff[ord(b_) - ord('a')] -= 1
        view = [diff[ord(c) - ord('a')] for c in show]
        print(f"bước {step}: +{a} -{b_} -> diff[{show[0]}..{show[-1]}] = {view}")
    worst = max(range(26), key=lambda k: abs(diff[k]))
    print(f"Chênh lớn nhất: chữ {chr(97 + worst)!r} = {diff[worst]} -> {'True' if abs(diff[worst]) <= 3 else 'False'}")
    return all(abs(d) <= 3 for d in diff)

trace_diff("abcdeef", "abaaacc")

bước 0: +a -a -> diff[a..f] = [0, 0, 0, 0, 0, 0]
bước 1: +b -b -> diff[a..f] = [0, 0, 0, 0, 0, 0]
bước 2: +c -a -> diff[a..f] = [-1, 0, 1, 0, 0, 0]
bước 3: +d -a -> diff[a..f] = [-2, 0, 1, 1, 0, 0]
bước 4: +e -a -> diff[a..f] = [-3, 0, 1, 1, 1, 0]
bước 5: +e -c -> diff[a..f] = [-3, 0, 0, 1, 2, 0]
bước 6: +f -c -> diff[a..f] = [-3, 0, -1, 1, 2, 1]
Chênh lớn nhất: chữ 'a' = -3 -> True


True

In [11]:
trace_diff("aaaa", "bccb", show="abc")

bước 0: +a -b -> diff[a..c] = [1, -1, 0]
bước 1: +a -c -> diff[a..c] = [2, -1, -1]
bước 2: +a -c -> diff[a..c] = [3, -1, -2]
bước 3: +a -b -> diff[a..c] = [4, -2, -2]
Chênh lớn nhất: chữ 'a' = 4 -> False


False

`max(range(26), key=lambda k: abs(diff[k]))` tìm chỉ số `k` có `abs(diff[k])` lớn nhất — `key` cho biết "so sánh theo tiêu chí nào".

## 7. Cách 4 — Phép trừ Counter hai chiều

Vì `c1 - c2` chỉ giữ phần dương, chênh lệch "word1 nhiều hơn" nằm ở `c1 - c2`, "word2 nhiều hơn" ở `c2 - c1`. Điều kiện: mọi giá trị trong cả hai đều ≤ 3.

In [12]:
def almost_equiv_subtract(word1, word2):
    c1, c2 = Counter(word1), Counter(word2)
    return all(v <= 3 for v in ((c1 - c2) + (c2 - c1)).values())

print(almost_equiv_subtract("aaaa", "bccb"))
print(almost_equiv_subtract("abcdeef", "abaaacc"))
print(almost_equiv_subtract("cccddabba", "babababab"))

False
True
True


Cộng hai Counter `(c1 - c2) + (c2 - c1)` gộp các key; vì một chữ không thể vừa dương ở bên này vừa dương ở bên kia, kết quả là `|freq1 − freq2|` cho mọi chữ có chênh lệch khác 0.

Một cách viết khác: `Counter.subtract` (trừ tại chỗ, **giữ cả số âm**):

In [13]:
c = Counter("abcdeef")
c.subtract("abaaacc")          # giữ giá trị âm, khác với toán tử '-'
print(c)
print(all(abs(v) <= 3 for v in c.values()))

Counter({'e': 2, 'd': 1, 'f': 1, 'b': 0, 'c': -1, 'a': -3})
True


## 8. Code nộp LeetCode

In [14]:
class Solution:
    def checkAlmostEquivalent(self, word1: str, word2: str) -> bool:
        diff = [0] * 26
        for a, b in zip(word1, word2):
            diff[ord(a) - ord('a')] += 1
            diff[ord(b) - ord('a')] -= 1
        return all(abs(d) <= 3 for d in diff)

sol = Solution()
print(sol.checkAlmostEquivalent("abcdeef", "abaaacc"))

True


Bản Counter (cũng nộp được):

In [15]:
class SolutionCounter:
    def checkAlmostEquivalent(self, word1: str, word2: str) -> bool:
        c = Counter(word1)
        c.subtract(word2)
        return all(abs(v) <= 3 for v in c.values())

print(SolutionCounter().checkAlmostEquivalent("aaaa", "bccb"))

False


## 9. Vì sao đúng?

**Invariant** khi duyệt cách 3: sau `t` bước, `diff[k]` = (số lần chữ `k` trong `word1[:t]`) − (số lần chữ `k` trong `word2[:t]`).

- Ban đầu `t = 0`: mọi `diff[k] = 0` ✓.
- Bước `t`: chữ `word1[t]` được `+1`, chữ `word2[t]` được `−1` — đúng bằng thay đổi của hai tần suất ✓ (nếu hai chữ trùng nhau thì `+1 −1 = 0`, cũng đúng).
- Sau `n` bước: `diff[k] = freq1[k] − freq2[k]` cho **mọi** 26 chữ, kể cả chữ chỉ có ở một chuỗi (vì mảng có đủ 26 ô, khởi tạo 0).

Vậy `all(abs(d) <= 3 for d in diff)` chính xác là định nghĩa "almost equivalent".

`zip` an toàn vì đề đảm bảo `len(word1) == len(word2)`; nếu hai độ dài khác nhau thì `zip` dừng sớm và **bỏ sót** ký tự (xem biến thể).

## 10. Độ phức tạp

Gọi `n` là độ dài mỗi chuỗi, `Σ = 26` là kích thước bảng chữ cái.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. 26 × `count` | `O(Σ · n)` | `O(1)` | đơn giản nhất |
| 2. Hai Counter + hợp key | `O(n)` trung bình | `O(Σ)` | phải nhớ hợp key |
| 3. Mảng hiệu 26 ô | `O(n + Σ)` | `O(Σ)` = `O(1)` | lời giải chuẩn |
| 4. Counter trừ hai chiều / subtract | `O(n)` trung bình | `O(Σ)` | ngắn |

## 11. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<38} PASS ({actual})")

cases = [
    ("ví dụ 1", "aaaa", "bccb", False),
    ("ví dụ 2", "abcdeef", "abaaacc", True),
    ("ví dụ 3", "cccddabba", "babababab", True),
    ("n = 1", "a", "b", True),
    ("giống hệt", "abc", "abc", True),
    ("chênh đúng 3 (biên)", "aaab", "bbbb", True),
    ("chênh 4 ở word2 (chữ chỉ có ở word2)", "abcd", "zzzz", False),
    ("chênh 4 ở word1", "zzzz", "abcd", False),
    ("nhiều chữ chênh 3", "aaabbb", "cccddd", True),
]
fs = [almost_equiv_count, almost_equiv_counter, almost_equiv_diff, almost_equiv_subtract,
      Solution().checkAlmostEquivalent, SolutionCounter().checkAlmostEquivalent]
for name, a, b_, exp in cases:
    for f in fs:
        assert f(a, b_) == exp, (f, a, b_)
    check(name, sol.checkAlmostEquivalent(a, b_), exp)

ví dụ 1                                PASS (False)
ví dụ 2                                PASS (True)
ví dụ 3                                PASS (True)
n = 1                                  PASS (True)
giống hệt                              PASS (True)
chênh đúng 3 (biên)                    PASS (True)
chênh 4 ở word2 (chữ chỉ có ở word2)   PASS (False)
chênh 4 ở word1                        PASS (False)
nhiều chữ chênh 3                      PASS (True)


In [17]:
import random
random.seed(2068)
for _ in range(2000):
    n = random.randint(1, 20)
    a = "".join(random.choice("abcd") for _ in range(n))
    b_ = "".join(random.choice("abcd") for _ in range(n))
    answers = {f(a, b_) for f in fs}
    assert len(answers) == 1, (a, b_)
print("2000 cặp chuỗi ngẫu nhiên: 6 cài đặt khớp nhau — PASS")

2000 cặp chuỗi ngẫu nhiên: 6 cài đặt khớp nhau — PASS


Bảng chữ nhỏ `"abcd"` làm tần suất mỗi chữ cao → nhiều ca chênh 3, 4 sát biên — đúng chỗ dễ sai.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ duyệt key của `word1`

In [18]:
def wrong_only_word1_keys(word1, word2):
    c1, c2 = Counter(word1), Counter(word2)
    for ch in c1:                         # bỏ sót chữ chỉ có ở word2
        if abs(c1[ch] - c2[ch]) > 3:
            return False
    return True

print("Sai :", wrong_only_word1_keys("abcd", "zzzz"))    # True (sai)
print("Đúng:", almost_equiv_diff("abcd", "zzzz"))        # False

Sai : True
Đúng: False


Đây chính là cảnh báo trong gợi ý của đề: *"Are there edge cases where a character is present in one string but not the other?"*

### Lỗi 2: Dùng `c1 - c2` một chiều

`c1 - c2` bỏ phần âm, nên chỉ phát hiện "word1 nhiều hơn".

In [19]:
def wrong_one_way_subtract(word1, word2):
    return all(v <= 3 for v in (Counter(word1) - Counter(word2)).values())

print("Sai :", wrong_one_way_subtract("abcd", "zzzz"))
print("Đúng:", almost_equiv_subtract("abcd", "zzzz"))

Sai : True
Đúng: False


### Lỗi 3: Quên `abs`, chỉ kiểm tra `diff > 3`

Bỏ sót trường hợp `word2` nhiều hơn (diff âm, vd −4).

### Lỗi 4: Dùng `< 3` thay vì `<= 3`

"At most 3" = **không quá 3**, tức `<= 3`. Chênh đúng 3 vẫn hợp lệ (ví dụ 2, 3 của đề).

In [20]:
def wrong_no_abs(word1, word2):
    diff = [0] * 26
    for a, b_ in zip(word1, word2):
        diff[ord(a) - 97] += 1
        diff[ord(b_) - 97] -= 1
    return all(d <= 3 for d in diff)

def wrong_strict(word1, word2):
    diff = [0] * 26
    for a, b_ in zip(word1, word2):
        diff[ord(a) - 97] += 1
        diff[ord(b_) - 97] -= 1
    return all(abs(d) < 3 for d in diff)

print("Quên abs :", wrong_no_abs("abcd", "zzzz"), "| đúng:", almost_equiv_diff("abcd", "zzzz"))
print("Dùng <3  :", wrong_strict("abcdeef", "abaaacc"), "| đúng:", almost_equiv_diff("abcdeef", "abaaacc"))

Quên abs : True | đúng: False
Dùng <3  : False | đúng: True


### Lỗi 5: Hiểu nhầm "tổng chênh lệch ≤ 3"

Đề nói chênh lệch **của từng chữ** ≤ 3, không phải **tổng** các chênh lệch. Ví dụ 2 có tổng chênh `3+0+1+1+2+1 = 8` nhưng vẫn `True`.

## 13. Mẹo nhận dạng pattern

- "So sánh **tần suất ký tự** giữa hai chuỗi" → mảng 26 ô (chữ thường) hoặc `Counter`.
- Chỉ cần **hiệu** tần suất → **một** mảng: `+1` cho chuỗi 1, `−1` cho chuỗi 2.
- Luôn tự hỏi: "**chữ chỉ có ở một bên** thì sao?" → duyệt đủ bảng chữ cái hoặc hợp key.
- Họ bài: 242 Valid Anagram (mọi `diff == 0`), 383 Ransom Note (`diff` một chiều ≥ 0), 1347 Minimum Number of Steps to Make Two Strings Anagram (tổng phần dương), 2287 Rearrange Characters to Make Target String.

## 14. Biến thể

1. **Ngưỡng `k` tổng quát:** `abs(d) <= k`. Với `k = 0` chính là **anagram** (LeetCode 242).
2. **Hai chuỗi khác độ dài:** không dùng `zip` — duyệt riêng từng chuỗi.
3. **Số bước tối thiểu để biến thành anagram** (LeetCode 1347): `sum(d for d in diff if d > 0)`.

In [21]:
def almost_equiv_k(word1, word2, k):
    diff = [0] * 26
    for a in word1:                       # duyệt riêng: an toàn khi độ dài khác nhau
        diff[ord(a) - 97] += 1
    for b_ in word2:
        diff[ord(b_) - 97] -= 1
    return all(abs(d) <= k for d in diff)

def is_anagram(s, t):
    return almost_equiv_k(s, t, 0)

def min_steps_anagram(s, t):
    diff = [0] * 26
    for a, b_ in zip(s, t):
        diff[ord(a) - 97] += 1
        diff[ord(b_) - 97] -= 1
    return sum(d for d in diff if d > 0)

print(is_anagram("listen", "silent"), is_anagram("ab", "abb"))
print(almost_equiv_k("aaaa", "bccb", 4))
print(min_steps_anagram("bab", "aba"), min_steps_anagram("leetcode", "practice"))

True False
True
1 5


## 15. Bài tập tự luyện

**Bài 1 — Tính tay:** với `word1 = "zzzab"`, `word2 = "abcde"`, tính `diff` cho các chữ `a, b, c, d, e, z` và kết luận.

**Bài 2 — Đoán output:** `ord('d') - ord('a')`, `chr(97 + 25)`, `Counter("aab") - Counter("abbb")`, `[0] * 3`.

**Bài 3 — TODO:** viết `exercise_almost_equiv(word1, word2)` bằng mảng hiệu 26 ô.

**Bài 4 — TODO:** `max_freq_gap(word1, word2)` trả về **chênh lệch tần suất lớn nhất** và **chữ** đạt giá trị đó (nếu nhiều chữ cùng lớn nhất, lấy chữ nhỏ nhất theo bảng chữ cái).

**Bài 5 — TODO biến thể:** `is_anagram_exercise(s, t)` — dùng mảng hiệu, xử lý cả khi `len(s) != len(t)`.

**Bài 6 — Mini challenge:** `count_almost_equiv_pairs(words, k)` — đếm số cặp `(i, j)`, `i < j`, mà `words[i]` và `words[j]` almost equivalent với ngưỡng `k`. Tiền xử lý mảng đếm mỗi từ **một lần**.

In [22]:
def exercise_almost_equiv(word1, word2):
    # TODO 1: diff = [0] * 26
    # TODO 2: +1 cho word1, -1 cho word2
    # TODO 3: all(abs(d) <= 3 ...)
    pass

# assert exercise_almost_equiv("aaaa", "bccb") == False

In [23]:
def max_freq_gap(word1, word2):
    # TODO: trả (gap, letter)
    pass

# assert max_freq_gap("aaaa", "bccb") == (4, "a")

In [24]:
def is_anagram_exercise(s, t):
    # TODO
    pass

# assert is_anagram_exercise("anagram", "nagaram") == True
# assert is_anagram_exercise("ab", "a") == False

In [25]:
def count_almost_equiv_pairs(words, k):
    # TODO: freq = [đếm 26 ô cho mỗi từ]; so sánh từng cặp
    pass

# assert count_almost_equiv_pairs(["aaaa", "bccb", "aab"], 3) == 2

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | đếm từng chữ | z: 3 − 0 | a 0, b 0, c −1, d −1, e −1, z 3 → True |
| 2 | ord liên tiếp | Counter trừ bỏ số ≤ 0 | `3`, `'z'`, `Counter({'a': 1})`, `[0,0,0]` |
| 3 | mục 5 | `diff[ord(a) - ord('a')] += 1` | `return all(abs(d) <= 3 for d in diff)` |
| 4 | duyệt 26 chỉ số | so sánh `abs(diff[k])` | duyệt k tăng dần, chỉ cập nhật khi **lớn hơn hẳn** |
| 5 | độ dài khác → False ngay hoặc duyệt riêng | không dùng zip | `all(d == 0 for d in diff)` |
| 6 | tính `freq` của mỗi từ một lần | hai vòng `i < j` | so `max(abs(x - y) for x, y in zip(fi, fj)) <= k` |

## 17. Đáp án tham khảo

**Bài 1:** `a: 1−1 = 0`, `b: 1−1 = 0`, `c: 0−1 = −1`, `d: −1`, `e: −1`, `z: 3−0 = 3` → mọi `|diff| ≤ 3` → **True**.

**Bài 2:** `3` · `'z'` · `Counter({'a': 1})` · `[0, 0, 0]`.

In [26]:
trace_diff("zzzab", "abcde", show="abcdez")
print(ord('d') - ord('a'), chr(97 + 25), Counter("aab") - Counter("abbb"), [0] * 3)

bước 0: +z -a -> diff[a..z] = [-1, 0, 0, 0, 0, 1]
bước 1: +z -b -> diff[a..z] = [-1, -1, 0, 0, 0, 2]
bước 2: +z -c -> diff[a..z] = [-1, -1, -1, 0, 0, 3]
bước 3: +a -d -> diff[a..z] = [0, -1, -1, -1, 0, 3]
bước 4: +b -e -> diff[a..z] = [0, 0, -1, -1, -1, 3]
Chênh lớn nhất: chữ 'z' = 3 -> True
3 z Counter({'a': 1}) [0, 0, 0]


In [27]:
def exercise_almost_equiv_answer(word1, word2):
    diff = [0] * 26
    for a, b_ in zip(word1, word2):
        diff[ord(a) - ord('a')] += 1
        diff[ord(b_) - ord('a')] -= 1
    return all(abs(d) <= 3 for d in diff)

def max_freq_gap_answer(word1, word2):
    diff = [0] * 26
    for a in word1:
        diff[ord(a) - 97] += 1
    for b_ in word2:
        diff[ord(b_) - 97] -= 1
    best_k = 0
    for k in range(26):
        if abs(diff[k]) > abs(diff[best_k]):
            best_k = k
    return abs(diff[best_k]), chr(97 + best_k)

def is_anagram_exercise_answer(s, t):
    if len(s) != len(t):
        return False
    diff = [0] * 26
    for a, b_ in zip(s, t):
        diff[ord(a) - 97] += 1
        diff[ord(b_) - 97] -= 1
    return all(d == 0 for d in diff)

def count_almost_equiv_pairs_answer(words, k):
    freqs = []
    for w in words:
        f = [0] * 26
        for ch in w:
            f[ord(ch) - 97] += 1
        freqs.append(f)
    total = 0
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if max(abs(x - y) for x, y in zip(freqs[i], freqs[j])) <= k:
                total += 1
    return total

assert exercise_almost_equiv_answer("aaaa", "bccb") == False
assert max_freq_gap_answer("aaaa", "bccb") == (4, "a")
assert is_anagram_exercise_answer("anagram", "nagaram") and not is_anagram_exercise_answer("ab", "a")
assert count_almost_equiv_pairs_answer(["aaaa", "bccb", "aab"], 3) == 2
random.seed(6)
for _ in range(500):
    n = random.randint(1, 12)
    a = "".join(random.choice("abc") for _ in range(n))
    b_ = "".join(random.choice("abc") for _ in range(n))
    assert exercise_almost_equiv_answer(a, b_) == almost_equiv_count(a, b_)
    gap = max(abs(a.count(c) - b_.count(c)) for c in string.ascii_lowercase)
    assert max_freq_gap_answer(a, b_)[0] == gap
    assert is_anagram_exercise_answer(a, b_) == (sorted(a) == sorted(b_))
print("Tất cả đáp án bài tập — PASS")

Tất cả đáp án bài tập — PASS


Ở bài 6, cặp `("aaaa","aab")` chênh `a`: 2, `b`: 1 → hợp lệ; `("bccb","aab")`: `a` 2, `b` 1, `c` 2 → hợp lệ; `("aaaa","bccb")` chênh 4 → không. Tổng 2.

## 18. Tổng kết — template + checklist

**Template "so sánh tần suất hai chuỗi":**

```python
diff = [0] * 26
for ch in s: diff[ord(ch) - ord('a')] += 1
for ch in t: diff[ord(ch) - ord('a')] -= 1
return all(abs(d) <= K for d in diff)      # K = 0: anagram; K = 3: bài này
```

**Checklist submit**

- [ ] Xét **mọi** 26 chữ (kể cả chữ chỉ có ở một chuỗi).
- [ ] Dùng `abs` (chênh lệch hai chiều).
- [ ] So sánh `<= 3` (không quá 3), không phải `< 3`.
- [ ] Điều kiện là **từng chữ**, không phải tổng.
- [ ] Tên method `checkAlmostEquivalent`, trả `bool`.